# ARC-AGI Latent Program JEPA (LP-JEPA)

In [1]:
import json, math, os, random, copy, glob
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__)

device: cuda | torch: 2.8.0+cu128


In [2]:
import subprocess, os
ARC_DIR = Path("data/arc")
if not ARC_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/fchollet/ARC-AGI.git", str(ARC_DIR)],
        check=True,
    )
print("train tasks:", len(list((ARC_DIR / "data/training").glob("*.json"))))
print("eval  tasks:", len(list((ARC_DIR / "data/evaluation").glob("*.json"))))

train tasks: 400
eval  tasks: 400


In [3]:
def load_arc_split(split):
    """Return dict task_id -> {'train': [(in,out),...], 'test': [(in,out),...]}."""
    out = {}
    for p in sorted((ARC_DIR / "data" / split).glob("*.json")):
        d = json.loads(p.read_text())
        out[p.stem] = {
            "train": [(np.array(e["input"]), np.array(e["output"])) for e in d["train"]],
            "test":  [(np.array(e["input"]), np.array(e["output"])) for e in d["test"]],
        }
    return out

TRAIN_TASKS = load_arc_split("training")
EVAL_TASKS  = load_arc_split("evaluation")
print(len(TRAIN_TASKS), len(EVAL_TASKS))

400 400


In [4]:
assert len(TRAIN_TASKS) == 400 and len(EVAL_TASKS) == 400
g = next(iter(TRAIN_TASKS.values()))["train"][0][0]
assert g.ndim == 2 and g.min() >= 0 and g.max() <= 9
print("OK: 400/400 tasks, grids are 2D int 0-9")

OK: 400/400 tasks, grids are 2D int 0-9


## Tokenizer

In [5]:
MAX_H = MAX_W = 30
NUM_COLORS = 10
PAD_ID = 10           # padding/color sentinel
VOCAB = 11            # colors 0..9 + PAD
N_CELLS = MAX_H * MAX_W          # 900
N_TOKENS = N_CELLS                # cells only; shape carried separately
D = 256

def pad_grid(grid):
    """np (h,w) int -> (padded (30,30) long, h, w). Out-of-grid cells = PAD_ID."""
    h, w = grid.shape
    assert 1 <= h <= MAX_H and 1 <= w <= MAX_W, f"grid {h}x{w} too big"
    out = np.full((MAX_H, MAX_W), PAD_ID, dtype=np.int64)
    out[:h, :w] = grid
    return torch.from_numpy(out), h, w

def grid_to_tokens(grid):
    """np (h,w) -> dict(tokens (900,), pad_mask (900,) True=pad, h, w)."""
    padded, h, w = pad_grid(grid)
    tokens = padded.reshape(-1)            # (900,)
    pad_mask = tokens.eq(PAD_ID)           # True where padding
    return {"tokens": tokens, "pad_mask": pad_mask,
            "h": torch.tensor(h), "w": torch.tensor(w)}

def tokens_to_grid(tokens, h, w):
    """tokens (900,) + shape -> np (h,w) int. Inverse of grid_to_tokens."""
    grid = tokens.reshape(MAX_H, MAX_W)[:h, :w]
    return grid.cpu().numpy().astype(np.int64)


In [6]:
ok = 0
for t in list(TRAIN_TASKS.values())[:50]:
    for (gi, go) in t["train"]:
        for g in (gi, go):
            enc = grid_to_tokens(g)
            rt = tokens_to_grid(enc["tokens"], int(enc["h"]), int(enc["w"]))
            assert np.array_equal(rt, g), "round-trip mismatch"
            ok += 1
print(f"OK: {ok} grids round-tripped identically")


OK: 330 grids round-tripped identically


## Augmentation & batch protocol

In [7]:
def dihedral(grid, k):
    """k in 0..7: 4 rotations x optional flip. Returns np grid."""
    g = np.rot90(grid, k % 4)
    if k >= 4:
        g = np.fliplr(g)
    return np.ascontiguousarray(g)

def color_perm(grid, perm):
    """perm: length-10 array, a permutation of colors 0..9. Background 0 kept fixed."""
    return perm[grid]

def random_perm(keep_bg=True):
    p = np.arange(NUM_COLORS)
    rest = p[1:] if keep_bg else p
    np.random.shuffle(rest)
    if keep_bg:
        p = np.concatenate([[0], rest])
    else:
        p = rest
    return p

def augment_pair(inp, out, k=None, perm=None):
    """Apply the SAME dihedral + color perm to both halves of a pair."""
    if k is None: k = random.randint(0, 7)
    if perm is None: perm = random_perm()
    return color_perm(dihedral(inp, k), perm), color_perm(dihedral(out, k), perm)


In [8]:
def encode_pair_grids(pair):
    """(in,out) np -> two token dicts."""
    gi, go = pair
    return grid_to_tokens(gi), grid_to_tokens(go)

def sample_episode(task, n_demo=None, augment=True):
    """Build one training episode from a task.
    Returns: demos (list of (in_tok,out_tok)), query (in_tok,out_tok).
    Query is one held-out pair; demos are the rest. With augmentation, a shared
    (k,perm) is drawn per episode so the program is invariant to it."""
    pairs = list(task["train"])
    random.shuffle(pairs)
    q = pairs[0]
    demos = pairs[1:]
    if n_demo is not None:
        demos = demos[:n_demo]
    if augment:
        k, perm = random.randint(0, 7), random_perm()
        demos = [augment_pair(i, o, k, perm) for (i, o) in demos]
        q = augment_pair(q[0], q[1], k, perm)
    demos_tok = [encode_pair_grids(p) for p in demos]
    q_tok = encode_pair_grids(q)
    return demos_tok, q_tok

def make_views(task, n_views=16):
    """Many augmented (in,out) token pairs of a task, for SIGReg batch dimension."""
    pairs = list(task["train"])
    views = []
    for _ in range(n_views):
        i, o = random.choice(pairs)
        ai, ao = augment_pair(i, o)
        views.append(encode_pair_grids((ai, ao)))
    return views


In [9]:
def collate_tokens(token_dicts):
    """list of token dicts -> dict of stacked tensors on DEVICE."""
    return {
        "tokens":   torch.stack([t["tokens"]   for t in token_dicts]).to(DEVICE),
        "pad_mask": torch.stack([t["pad_mask"] for t in token_dicts]).to(DEVICE),
        "h":        torch.stack([t["h"]        for t in token_dicts]).to(DEVICE),
        "w":        torch.stack([t["w"]        for t in token_dicts]).to(DEVICE),
    }


In [10]:
task = next(iter(TRAIN_TASKS.values()))
demos, q = sample_episode(task)
assert len(demos) >= 1 and len(q) == 2
b = collate_tokens([d[0] for d in demos])
assert b["tokens"].shape[1] == N_CELLS and b["pad_mask"].dtype == torch.bool
v = make_views(task, n_views=8)
assert len(v) == 8
print("OK: episodes, views, collation shapes valid")


OK: episodes, views, collation shapes valid


## Grid encoder E

In [11]:
class GridEncoder(nn.Module):
    """ViT-style transformer over 30x30 color tokens -> pooled latent (B, D)."""
    def __init__(self, d=D, depth=4, heads=4, mlp_mult=4):
        super().__init__()
        self.color_emb = nn.Embedding(VOCAB, d)
        self.row_emb = nn.Embedding(MAX_H, d)
        self.col_emb = nn.Embedding(MAX_W, d)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.normal_(self.cls, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, d * mlp_mult,
                                           batch_first=True, norm_first=True,
                                           activation="gelu")
        self.tf = nn.TransformerEncoder(layer, depth)
        self.norm = nn.LayerNorm(d)
        rows = torch.arange(MAX_H).repeat_interleave(MAX_W)   # (900,)
        cols = torch.arange(MAX_W).repeat(MAX_H)              # (900,)
        self.register_buffer("rows", rows)
        self.register_buffer("cols", cols)

    def forward(self, tokens, pad_mask):
        # tokens (B,900) long, pad_mask (B,900) True=pad
        B = tokens.size(0)
        x = self.color_emb(tokens) + self.row_emb(self.rows) + self.col_emb(self.cols)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)                       # (B, 901, D)
        # CLS never masked
        mask = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=tokens.device),
                          pad_mask], dim=1)
        x = self.tf(x, src_key_padding_mask=mask)
        return self.norm(x[:, 0])                            # (B, D)


In [12]:
E = GridEncoder().to(DEVICE)
demos, q = sample_episode(next(iter(TRAIN_TASKS.values())))
qb = collate_tokens([q[0]])
a = E(qb["tokens"], qb["pad_mask"])
assert a.shape == (1, D) and torch.isfinite(a).all()
print("OK: GridEncoder ->", tuple(a.shape))


c:\Users\Ous\miniconda3\envs\ML\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


OK: GridEncoder -> (1, 256)


## Program encoder P

In [13]:
class ProgramEncoder(nn.Module):
    """Encode one (input,output) demo pair -> z_i (B, D). Mean over pairs gives z."""
    def __init__(self, d=D, depth=4, heads=4, mlp_mult=4):
        super().__init__()
        self.color_emb = nn.Embedding(VOCAB, d)
        self.row_emb = nn.Embedding(MAX_H, d)
        self.col_emb = nn.Embedding(MAX_W, d)
        self.seg_emb = nn.Embedding(2, d)            # 0=input half, 1=output half
        self.cls = nn.Parameter(torch.zeros(1, 1, d)); nn.init.normal_(self.cls, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, d * mlp_mult,
                                           batch_first=True, norm_first=True,
                                           activation="gelu")
        self.tf = nn.TransformerEncoder(layer, depth)
        self.norm = nn.LayerNorm(d)
        rows = torch.arange(MAX_H).repeat_interleave(MAX_W)
        cols = torch.arange(MAX_W).repeat(MAX_H)
        self.register_buffer("rows", rows); self.register_buffer("cols", cols)

    def _embed(self, tokens, seg):
        return (self.color_emb(tokens) + self.row_emb(self.rows)
                + self.col_emb(self.cols) + self.seg_emb.weight[seg])

    def forward(self, in_tok, in_mask, out_tok, out_mask):
        # each (B,900). Concatenate input and output halves into one sequence.
        B = in_tok.size(0)
        xi = self._embed(in_tok, 0)
        xo = self._embed(out_tok, 1)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, xi, xo], dim=1)               # (B, 1801, D)
        mask = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=in_tok.device),
                          in_mask, out_mask], dim=1)
        x = self.tf(x, src_key_padding_mask=mask)
        return self.norm(x[:, 0])                          # (B, D)

def encode_program(P, demos_tok, exclude=None):
    """Mean of per-pair latents over a list of (in_tok,out_tok). Leave-one-out via
    `exclude` (index to drop). Returns (1, D)."""
    idxs = [i for i in range(len(demos_tok)) if i != exclude]
    ins  = collate_tokens([demos_tok[i][0] for i in idxs])
    outs = collate_tokens([demos_tok[i][1] for i in idxs])
    z_i = P(ins["tokens"], ins["pad_mask"], outs["tokens"], outs["pad_mask"])  # (k, D)
    return z_i.mean(0, keepdim=True)                                            # (1, D)

In [14]:
P = ProgramEncoder().to(DEVICE)
demos, q = sample_episode(next(iter(TRAIN_TASKS.values())))
z = encode_program(P, demos)
assert z.shape == (1, D)
z_loo = encode_program(P, demos, exclude=0)   # leave-one-out path runs
assert z_loo.shape == (1, D)
print("OK: ProgramEncoder + mean/LOO ->", tuple(z.shape))

OK: ProgramEncoder + mean/LOO -> (1, 256)


c:\Users\Ous\miniconda3\envs\ML\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
